# Конспект. Модуль 5: Матричная факторизация — скрытые факторы (SVD, ALS, BPR)

**Курс:** Мини-курс RecSys (13 модулей)
**Модуль:** 5 из 13 — «Матричная факторизация: скрытые факторы (SVD, ALS, BPR)»
**Цель модуля:** совершить переход от **memory-based** методов (Модули 3–4, где предсказание строится «на лету» через прямое сравнение сырых векторов) к **model-based** методам, где вместо хранения и сравнения полных векторов оценок обучается компактное представление каждого пользователя и товара. Это прямой ответ сразу на две проблемы, поднятые в предыдущих модулях: разреженность (Модуль 1.4.2) и вычислительную сложность (Модули 3.3.1, 4.2.1).

**Важное методологическое примечание:** все численные примеры в этом модуле рассчитаны и явно проверены выполнением реального кода (NumPy), а не вычислены вручную «на бумаге» — матричные разложения и итеративная оптимизация слишком чувствительны к ошибкам округления для надёжных ручных расчётов. Все приведённые числа воспроизводимы (указаны random seed и гиперпараметры).

## 5.1 Проблема размерности — зачем вообще переходить к новому классу методов

Напомним ключевые числа из Модуля 1.4.2 и Модуля 3.3.1: при 100 000 пользователей и 10 000 товаров матрица содержит `10^9` ячеек, из которых заполнено обычно не более 1%. Прямое сравнение пользователей (UB-CF) или товаров (IB-CF) в лучшем случае даёт `O(N²M)` или `O(M²N)` — счётно неудобные величины при промышленных масштабах.

**Идея матричной факторизации:** вместо того чтобы хранить и сравнивать разреженные векторы напрямую, сожмём всю доступную информацию о каждом пользователе и товаре в **компактный плотный вектор** фиксированной длины (`n_factors`, обычно 20–300 чисел, независимо от того, сколько всего товаров в каталоге — миллион или сто). Как только такие векторы обучены, предсказание для **любой** пары (пользователь, товар) — это одна операция скалярного произведения, `O(n_factors)`, вместо перебора соседей.

## 5.2 Идея скрытых факторов (Latent Factors)

### 5.2.1 Иллюстративный игрушечный пример

Прежде чем переходить к формальной математике, разберём идею на предельно простом примере с всего двумя воображаемыми «скрытыми измерениями» — условно назовём их «любит боевики» и «любит романтику» (в реальности такие факторы обучаются автоматически и обычно **не имеют** настолько чистой человеческой интерпретации, но для интуиции это полезное упрощение).

**Пользователи** (вектор `[любит_боевики, любит_романтику]`):

In [ ]:
U1 = [0.9, 0.1]   — явно любит боевики, равнодушен к романтике
U2 = [0.2, 0.8]   — явно любит романтику, равнодушен к боевикам
U3 = [0.6, 0.6]   — любит и то, и другое примерно поровну

**Товары** (вектор `[боевик_компонента, романтика_компонента]`):

In [ ]:
I1 (чистый боевик)          = [1.0, 0.0]
I2 (чистая романтика)       = [0.0, 1.0]
I3 (гибрид боевик+романтика) = [0.5, 0.5]

**Предсказание = скалярное произведение вектора пользователя и вектора товара:**

In [ ]:
pred(U1, I1) = 0.9×1.0 + 0.1×0.0 = 0.90   -> высокая оценка (ожидаемо: любитель боевиков + боевик)
pred(U1, I2) = 0.9×0.0 + 0.1×1.0 = 0.10   -> низкая оценка (ожидаемо: не любит романтику)
pred(U2, I1) = 0.2×1.0 + 0.8×0.0 = 0.20   -> низкая
pred(U2, I2) = 0.2×0.0 + 0.8×1.0 = 0.80   -> высокая
pred(U3, I3) = 0.6×0.5 + 0.6×0.5 = 0.60   -> умеренно высокая (сбалансированный пользователь + гибридный фильм)

На практике эти сырые скалярные произведения дополнительно масштабируются и сдвигаются bias-термами (раздел 5.4.2), чтобы попасть в реальный диапазон шкалы оценок (например, 1–5) — здесь показана только суть механизма.

### 5.2.2 Формальная постановка

- Каждому пользователю `u` сопоставляется вектор `p_u ∈ R^f` (`f` = число латентных факторов).
- Каждому товару `i` сопоставляется вектор `q_i ∈ R^f`.
- Предсказание: `pred(u, i) = p_u · q_i` (плюс bias-термы, раздел 5.4.2).
- Векторы `p_u` и `q_i` **не задаются вручную** (в отличие от игрушечного примера выше) — они **обучаются** так, чтобы как можно точнее восстанавливать известные, реально наблюдаемые оценки. Разные разделы этого модуля (5.3–5.6) — это разные способы **обучить** эти векторы.

## 5.3 SVD (Singular Value Decomposition)

### 5.3.1 Математическая формулировка

Любую матрицу `R` размера `n × m` можно разложить как:

In [ ]:
R = U · Σ · Vᵀ

где:
- `U` (`n × n`) — ортогональная матрица, столбцы которой являются собственными векторами `R·Rᵀ` («пользователи в скрытом пространстве»),
- `V` (`m × m`) — ортогональная матрица, столбцы которой являются собственными векторами `Rᵀ·R` («товары в скрытом пространстве»),
- `Σ` (`n × m`) — диагональная матрица **сингулярных чисел**, отсортированных по убыванию, где `σ_k = √λ_k` (`λ_k` — собственные значения `Rᵀ·R`).

Это прямое применение вашей уже сильной базы по линейной алгебре: сингулярное разложение всегда существует для любой матрицы (в отличие от, например, собственного разложения, которое требует квадратности и не всегда возможно) — это одна из причин его фундаментальной важности не только в RecSys, но и в анализе данных в целом (PCA математически эквивалентен SVD на центрированных данных).

### 5.3.2 Полностью проверенный численный пример

Возьмём небольшую **полностью заполненную** матрицу оценок (4 пользователя × 3 фильма), специально сконструированную так, чтобы в ней прослеживались две группы вкусов — пользователи 1-2 любят фильмы 1-2 и не любят фильм 3, пользователи 3-4 — наоборот:

In [ ]:
R = [[5, 4, 1],
     [4, 5, 1],
     [1, 1, 5],
     [1, 2, 4]]

**Результат `np.linalg.svd(R)`:**

In [ ]:
U (4×3) =
[[-0.6100  0.3577 -0.6455]
 [-0.6136  0.3401  0.5890]
 [-0.3436 -0.7125 -0.3069]
 [-0.3651 -0.4988  0.3772]]

Σ (сингулярные числа) = [10.0650, 5.4213, 1.1426]

Vᵀ (3×3) =
[[-0.6173 -0.6540 -0.4373]
 [ 0.3574  0.2621 -0.8964]
 [-0.7009  0.7097 -0.0719]]

**Проверка связи с собственными значениями (ваша сильная сторона — линейная алгебра):**

In [ ]:
Собственные значения RᵀR = [101.3040, 29.3905, 1.3055]
Σ² =                        [101.3040, 29.3905, 1.3055]   <- точное совпадение

Это не совпадение, а прямое следствие определения: `σ_k² = λ_k(RᵀR)`.

### 5.3.3 Усечённое SVD (Truncated SVD) — сжатие размерности

Ключевая практическая идея: если оставить только первые `k` (самых больших) сингулярных чисел и соответствующие столбцы `U` и `V`, получится **наилучшее возможное приближение** матрицы `R` рангом `k` (теорема Эккарта-Янга) — то есть ни одна другая матрица ранга `k` не приблизит `R` точнее в смысле нормы Фробениуса.

**Проверенные результаты усечения для нашего примера:**

| k | Объяснённая доля "энергии" матрицы | Реконструкция (округлённо) |
|:---:|:---:|:---|
| 1 | 76.75% | `[[3.79,4.02,2.69],[3.81,4.04,2.70],[2.13,2.26,1.51],[2.27,2.40,1.61]]` |
| 2 | 99.01% | `[[4.48,4.52,0.95],[4.47,4.52,1.05],[0.75,1.25,4.97],[1.30,1.69,4.03]]` |
| 3 (полный ранг) | 100.00% | точное восстановление исходной `R` |

**Интерпретация:** уже при `k=2` (всего 2 латентных фактора вместо исходных 3 измерений-товаров) восстановленная матрица объясняет **99.01%** дисперсии исходных данных и визуально почти неотличима от оригинала (сравните `4.48` с исходным `5`, `0.95` с исходным `1` — совпадение по порядку величины и, что важнее, **по относительным различиям между пользователями и товарами**). Это именно то сжатие информации, о котором говорилось в разделе 5.2: два числа на пользователя вместо трёх (а в реальных системах — 50-300 вместо миллионов) уже улавливают почти всю содержательную структуру данных.

### 5.3.4 Проблема классического SVD — пропуски в данных

Приведённый выше пример работал с **полностью заполненной** матрицей. Но user-item матрица в реальности разрежена на 95-99% (Модуль 1.4.2) — что делать с пропусками?

**Наивное (и неверное) решение:** заполнить пропуски нулями или средним значением перед применением SVD. Проблема: ноль в контексте рейтинга по шкале 1-5 — это **не «нейтральная» информация**, это ложный сигнал «пользователь оценил бы это на 0», который исказит всё разложение, искусственно утягивая скрытые векторы в сторону заниженных предсказаний именно для тех пар (пользователь, товар), где на самом деле просто нет данных, а не есть настоящая низкая оценка.

**Правильное решение:** не пытаться применить формулу SVD напрямую к неполной матрице, а **обучить** аналог `U` и `V` (в этом контексте их обычно называют `P` и `Q`, чтобы подчеркнуть, что это уже не строгое SVD-разложение) методом, который **учитывает только реально известные ячейки**, полностью игнорируя пропуски на этапе оптимизации. Это и есть Funk SVD (следующий раздел) — практический ответ на теоретическое ограничение классического SVD.

## 5.4 Funk SVD и обучение через градиентный спуск

### 5.4.1 Постановка задачи оптимизации

Вместо аналитического разложения, определим функцию потерь **только по известным ячейкам** и минимизируем её градиентным спуском:

In [ ]:
L = Σ_(u,i)∈known (r_ui - p_u·q_i)² + λ(||p_u||² + ||q_i||²)

Регуляризационный член `λ(...)` — та же самая идея L2/Ridge-регуляризации, с которой вы уже работали применительно к логистической регрессии (Неделя 4 общего плана): без неё латентные векторы могут разрастись до сколь угодно больших значений, идеально подгоняясь под шум обучающих данных.

### 5.4.2 Вывод формулы обновления (градиентный спуск, шаг за шагом)

Распишем производную по `p_u` для одной наблюдаемой ячейки `(u,i)` с ошибкой `e_ui = r_ui - p_u·q_i`:

In [ ]:
∂L/∂p_u = -2·e_ui·q_i + 2λ·p_u
∂L/∂q_i = -2·e_ui·p_u + 2λ·q_i

Шаг градиентного спуска (поглощая множитель 2 в learning rate `η`):

In [ ]:
p_u <- p_u + η(e_ui·q_i - λ·p_u)
q_i <- q_i + η(e_ui·p_u - λ·q_i)

**Важный практический нюанс реализации:** при обновлении `q_i` нужно использовать **старое** значение `p_u` (до его собственного обновления в этой же итерации), иначе обновления двух векторов будут несогласованными. В коде (раздел 5.7) это явно отражено сохранением `P_u_old` перед обновлением.

### 5.4.3 Bias-термы — практическое дополнение к чистому скалярному произведению

На практике почти никогда не используют «голое» скалярное произведение — добавляют bias-термы, учитывающие систематические смещения:

In [ ]:
pred(u, i) = μ + b_u + b_i + p_u·q_i

где `μ` — глобальное среднее по всем оценкам, `b_u` — систематическое смещение пользователя (щедрый/строгий — прямая связь с Модулем 2.3.1), `b_i` — систематическое смещение товара (объективно культовый фильм получает завышенные оценки независимо от того, кто его смотрит). Латентные векторы `p_u·q_i` в таком случае отвечают только за **персонализированную** часть предсказания, а не за общие смещения — это разделение ускоряет и стабилизирует обучение.

### 5.4.4 Полный проверенный численный пример

Используем сквозную матрицу курса (Модули 3-4):

| | I1 | I2 | I3 | I4 | I5 |
|:---|:---:|:---:|:---:|:---:|:---:|
| U1 | 5 | 3 | 4 | ? | — |
| U2 | 4 | 2 | 3 | 4 | — |
| U3 | 1 | 5 | 2 | 1 | 5 |
| U4 | 5 | 3 | 5 | 5 | — |
| U5 | 2 | 2 | 1 | — | 3 |

**Гиперпараметры:** `n_factors=2`, `learning_rate=0.05`, `λ=0.02`, случайная инициализация (`seed=42`), 20 известных рейтингов из 25 возможных ячеек.

**Динамика RMSE по эпохам (реально вычислено):**

| Эпоха | RMSE |
|:---:|:---:|
| 0 (до обучения) | 1.4473 |
| 1 | 1.3395 |
| 5 | 1.1112 |
| 10 | 0.8814 |
| 25 | 0.2119 |
| 50 | 0.1912 |
| 100 | 0.0493 |
| 250 | 0.0296 |
| 500 | 0.0284 |

**Итоговые предсказания для неизвестных ячеек:**

In [ ]:
pred(U1, I4) = 4.939
pred(U1, I5) = 4.323
pred(U2, I5) = 3.361
pred(U4, I5) = 4.243
pred(U5, I4) = 1.983

**Сравнение с предыдущими модулями для одной и той же ячейки U1-I4:**

| Метод | Предсказание для U1, I4 |
|:---|:---:|
| UB-CF (Модуль 3.2.3) | 5.000 (после клиппинга) |
| IB-CF, только положительные соседи (Модуль 4.3.4) | 4.566 |
| Funk SVD | 4.939 |

Все три принципиально разных алгоритма **согласованно** предсказывают, что U1 высоко оценит I4 — это ценная кросс-проверка (в реальном пайплайне такое согласие между независимыми моделями повышает доверие к предсказанию; несогласие, напротив, сигнализирует о неопределённости).

### 5.4.5 Критически важное наблюдение — переобучение в игрушечном примере

Обратите внимание на итоговое значение RMSE — **0.0284**, что означает практически идеальное восстановление всех известных рейтингов. Это не признак «отличной модели» — это явный **артефакт переобучения**, и вот почему: при `n_factors=2` модель содержит `5×2 (P) + 5×2 (Q) + 5 (b_u) + 5 (b_i) + 1 (μ) = 31` обучаемых параметров, что **больше**, чем `20` известных рейтингов в обучающих данных. Модель имеет достаточно «степеней свободы», чтобы буквально запомнить каждое известное значение, а не выучить обобщающуюся закономерность.

**Прямая связь с уже знакомой вам темой (Неделя 4, регуляризация):** на реальных данных, где число известных рейтингов исчисляется миллионами, а число параметров (даже при большом `f=200`) — тысячами или тем же порядком, это соотношение переворачивается в обратную сторону, и переобучение не является автоматическим — но сила регуляризации `λ` всё равно требует подбора через кросс-валидацию, а не выбирается «на глаз». Этот игрушечный пример — намеренно упрощённая иллюстрация механики обучения, а не образец правильного выбора гиперпараметров для реальной задачи.

## 5.5 ALS (Alternating Least Squares)

### 5.5.1 Идея — почему «alternating» и почему это не градиентный спуск

Ключевое наблюдение: если **зафиксировать** все векторы товаров `Q`, то задача нахождения оптимального `p_u` для конкретного пользователя превращается в обычную **регуляризованную линейную регрессию** (Ridge regression) — вы уже прекрасно знакомы с её закрытым аналитическим решением:

In [ ]:
p_u = (Qᵀ_u·Q_u + λI)^(-1) · Qᵀ_u·r_u

где `Q_u` — подматрица `Q`, содержащая только строки товаров, оценённых пользователем `u`, а `r_u` — вектор его оценок. Это **точно та же формула**, что и аналитическое решение Ridge-регрессии `(XᵀX + λI)^(-1)Xᵀy`, которую вы уже видели в общем ML-треке — только здесь роль «признаков» играют латентные векторы товаров.

**Алгоритм ALS:**
1. Зафиксировать `Q`, решить точную задачу наименьших квадратов для **каждого** `p_u` (аналитически, не градиентным спуском).
2. Зафиксировать (уже обновлённые) `P`, решить точную задачу для **каждого** `q_i`.
3. Повторять до сходимости.

**Почему это хорошо масштабируется:** на шаге 1 обновления всех `p_u` **полностью независимы друг от друга** — это идеальная задача для распараллеливания (каждый пользователь обрабатывается на отдельном ядре/машине, отсюда и историческая популярность Spark ALS для по-настоящему больших данных).

### 5.5.2 Полный проверенный численный пример (explicit ALS)

Та же матрица, что в разделе 5.4.4. Гиперпараметры: `n_factors=2`, `λ=0.1` (обратите внимание — регуляризация здесь **сильнее**, чем в примере Funk SVD выше, — это намеренный выбор для следующего наблюдения).

**Динамика RMSE по итерациям (реально вычислено):**

| Итерация | RMSE |
|:---:|:---:|
| до обучения | 3.556 |
| 1 | 0.798 |
| 2 | 0.405 |
| 3 | 0.251 |
| 5 | 0.236 |
| 10 | 0.235 |
| 15 | 0.235 |

**Ключевое наблюдение в сравнении с Funk SVD (5.4.5):** здесь RMSE стабилизируется на уровне **~0.235**, а не падает почти до нуля, как в примере Funk SVD (RMSE 0.028 при `λ=0.02`). Разница — именно в силе регуляризации (`λ=0.1` против `λ=0.02`): более сильная регуляризация здесь **намеренно не позволяет** модели идеально подогнаться под обучающие данные, оставляя «запас» для обобщения на новые, ещё не увиденные оценки. Это прямая иллюстрация bias-variance компромисса (тот же принцип, что вы уже видели применительно к выбору `k` в UB-CF, Модуль 3.3.5) — только здесь роль регулятора сложности играет `λ`, а не `k`.

**Итоговые предсказания:**

In [ ]:
pred(U1, I4) = 4.669 (совпадает по порядку с Funk SVD: 4.939, и с UB-CF/IB-CF)
pred(U1, I5) = 5.715 -> 5.000 после клиппинга
pred(U2, I5) = 4.254
pred(U4, I5) = 6.117 -> 5.000 после клиппинга
pred(U5, I4) = 1.592

### 5.5.3 Демонстрация одного явного шага закрытого решения (для U1)

Чтобы полностью развеять ощущение «магии» формулы, распишем конкретные числа для одного шага решения `p_{U1}` при уже (почти) сошедшихся векторах товаров:

**Товары, оценённые U1:** I1, I2, I3, с латентными векторами (после сходимости алгоритма):

In [ ]:
Q_u = [[-1.6432, -0.6503],
       [ 0.2471, -1.3426],
       [-1.1343, -0.7624]]

r_u = [5, 3, 4]

**Матрица нормальных уравнений (с регуляризацией):**

In [ ]:
A = Qᵀ_u·Q_u + λI = [[4.1478, 1.6016],
                      [1.6016, 2.9067]]

b = Qᵀ_u·r_u = [-12.0119, -10.3289]

**Решение системы `A·p_u = b`:**

In [ ]:
p_u = A^(-1)·b = [-1.9357, -2.4869]

Это ровно та же операция, что `sklearn.linear_model.Ridge` выполняет под капотом при вызове `.fit()` — только здесь «признаками» выступают латентные векторы товаров, а не заранее заданные фичи.

### 5.5.4 Implicit ALS — специфика для implicit feedback (Hu, Koren, Volinsky, 2008)

**Проблема, напомним из Модуля 1.3.2:** в implicit-данных нет явного негатива — «не взаимодействовал» не означает «не понравится бы», это может означать что угодно, включая «просто ещё не видел».

**Ключевая идея:** переопределить задачу через два разных понятия:
- **Предпочтение `p_ui ∈ {0, 1}`:** бинарный факт — было ли взаимодействие вообще.
- **Уверенность (confidence) `c_ui`:** насколько мы **доверяем** этому наблюдению.

In [ ]:
c_ui = 1 + α · r_ui

где `r_ui` — «сырой» сигнал силы взаимодействия (число просмотров, значение явного рейтинга, если оно есть, и т.д.), а `α` — гиперпараметр масштаба (в нашем примере — `α=2.0`).

**Принципиальное отличие от explicit ALS:** здесь оптимизация проводится **по всем парам (пользователь, товар)**, включая те, где взаимодействия не было (`p_ui=0`, но `c_ui=1` — базовый, минимальный уровень уверенности, а не «уверенность в нуле», это важное отличие в интерпретации). Формула для `p_u` модифицируется:

In [ ]:
p_u = (Qᵀ·C_u·Q + λI)^(-1) · Qᵀ·C_u·p(u)

где `C_u` — диагональная матрица уверенности пользователя `u` **по всем** товарам, а `p(u)` — вектор бинарных предпочтений по всем товарам.

**Вычислительный трюк (важно для реальных систем):** наивно эта формула требует работы с **полным** каталогом `M` товаров для **каждого** пользователя — что при миллионах товаров неприемлемо. Hu, Koren и Volinsky показали, что формулу можно переписать так, чтобы избежать этого:

In [ ]:
Qᵀ·C_u·Q = QᵀQ + Qᵀ(C_u - I)Q

Матрица `QᵀQ` считается **один раз** и переиспользуется для всех пользователей (`O(f²M)`, не зависит от числа пользователей), а поправочный член `Qᵀ(C_u-I)Q` — ненулевой **только** для товаров, с которыми пользователь реально взаимодействовал (там, где `c_ui ≠ 1`), то есть считается за время, пропорциональное числу **реальных** взаимодействий пользователя, а не всему каталогу. Именно этот трюк делает implicit ALS практически применимым в промышленных масштабах, и именно он реализован в библиотеке `implicit` (раздел 5.7).

### 5.5.5 Полный проверенный численный пример implicit ALS

Та же исходная матрица, но переинтерпретированная как implicit: `p_ui=1`, если ячейка была заполнена (независимо от значения), иначе `0`. Confidence: `c_ui = 1 + 2.0 × r_ui` (используя исходное значение рейтинга как «силу» сигнала; для незаполненных ячеек `r_ui=0 -> c_ui=1`).

**Динамика функции потерь по итерациям (реально вычислено):**

| Итерация | Loss |
|:---:|:---:|
| до обучения | 150.31 |
| 1 | 76.53 |
| 2 | 8.28 |
| 3 | 7.39 |
| 5 | 6.34 |
| 10 | 4.98 |
| 15 | 4.28 |

**Итоговый результат — ранжирование невзаимодействованных товаров для U1** (U1 не взаимодействовал с I4 и I5):

In [ ]:
I4: predicted score = 0.826
I5: predicted score = 0.317

**Интерпретация:** модель ранжирует I4 **выше** I5 для U1 — то есть, если бы нужно было порекомендовать U1 один из двух ещё не увиденных товаров, implicit ALS выбрал бы I4. Это **согласуется** со всеми предыдущими методами этого модуля (UB-CF, IB-CF, Funk SVD, explicit ALS), которые также предсказывали высокую оценку U1 для I4. Обратите внимание: в отличие от explicit-версий, здесь на выходе получается не «оценка по шкале 1-5», а **relevance score**, естественно приспособленный для задачи **ранжирования** — прямая связь с темой, которая станет центральной в Модуле 9 (Learning to Rank).

## 5.6 BPR (Bayesian Personalized Ranking)

### 5.6.1 Постановка задачи — почему не «ещё один способ предсказать число»

BPR принципиально меняет саму формулировку задачи. Вместо «предскажи предпочтение/рейтинг для пары (u, i)» ставится вопрос: «для пользователя `u`, который взаимодействовал с товаром `i`, но не взаимодействовал с товаром `j` — предскажи, что `i` для него **более предпочтителен**, чем `j`». Это классический **pairwise** подход (первое появление в курсе понятия, которое будет формализовано в Модуле 9.3 как общий класс методов Learning to Rank).

### 5.6.2 Функция потерь

In [ ]:
x_uij = p_u·q_i - p_u·q_j

L = -ln(σ(x_uij)) + λ(||p_u||² + ||q_i||² + ||q_j||²)

где `σ` — сигмоида. Минимизация `-ln(σ(x_uij))` эквивалентна максимизации `σ(x_uij)` — то есть максимизации вероятности того, что модель «правильно» предскажет `i` предпочтительнее `j` (`x_uij > 0`).

### 5.6.3 Вывод градиентов

Используем стандартное тождество производной логарифма сигмоиды: `d/dx[ln σ(x)] = σ(-x)`, следовательно `d/dx[-ln σ(x)] = -σ(-x)`.

По цепному правилу (`x_uij` зависит от `p_u`, `q_i`, `q_j` линейно):

In [ ]:
∂L/∂p_u = -σ(-x_uij)·(q_i - q_j) + λ·p_u
∂L/∂q_i = -σ(-x_uij)·p_u + λ·q_i
∂L/∂q_j =  σ(-x_uij)·p_u + λ·q_j

### 5.6.4 Полный проверенный численный пример одного шага обучения

Возьмём иллюстративные (не обученные) векторы: `p_u=[0.3,-0.2]` (U1), `q_i=[0.5,0.1]` (I1, позитив — U1 с ним взаимодействовал), `q_j=[-0.1,0.4]` (I5, негатив — U1 с ним не взаимодействовал). `η=0.1`, `λ=0.01`.

In [ ]:
x_uij = p_u·q_i - p_u·q_j = 0.1300 - (-0.1100) = 0.2400
σ(-x_uij) = 0.4403
Loss = -ln(σ(0.24)) = 0.5803

**Градиенты:**

In [ ]:
∂L/∂p_u = [-0.2612, 0.1301]
∂L/∂q_i = [-0.1271, 0.0891]
∂L/∂q_j = [ 0.1311, -0.0841]

**После одного шага SGD (`θ <- θ - η·∇L`):**

In [ ]:
p_u = [0.3261, -0.2130]
q_i = [0.5127, 0.0911]
q_j = [-0.1131, 0.4084]

Новое x_uij = 0.2717 (было 0.2400) -> выросло, как и требовалось
Новый Loss = 0.5665 (было 0.5803) -> уменьшился

**Интерпретация:** после одного шага обучения разница в оценке между позитивным (I1) и негативным (I5) товаром для U1 **увеличилась** — модель стала чуть более уверена в правильном относительном порядке. Это то самое прямое оптимизирование **порядка**, о котором говорилось в 5.6.1 — а не абсолютного значения рейтинга.

### 5.6.5 Когда BPR предпочтительнее ALS для implicit-данных

И implicit ALS (5.5.4), и BPR решают одну и ту же исходную проблему (implicit feedback без явного негатива), но по-разному:
- **Implicit ALS** оптимизирует **поточечную** (pointwise) ошибку по всем парам (пользователь, товар), используя confidence-взвешивание для компенсации отсутствия явного негатива.
- **BPR** напрямую оптимизирует **относительный порядок** внутри пар «взаимодействовал/не взаимодействовал», что часто ближе к тому, что реально нужно бизнесу (правильная последовательность в top-K), а не к точности предсказания «оценки».

На практике выбор между ними — предмет экспериментальной проверки на конкретных данных (обе модели доступны в библиотеке `implicit`, раздел 5.7), но концептуальное различие «pointwise vs pairwise» будет системно углубляться в Модуле 9.

## 5.7 Практика

### 5.7.1 Полная реализация Funk SVD с нуля (воспроизведение 5.4.4)

In [ ]:
import numpy as np

def train_funk_svd(R: np.ndarray, n_factors=2, lr=0.05, reg=0.02, n_epochs=500, seed=42):
    """
    R: матрица с np.nan на месте пропусков.
    Возвращает P, Q, b_u, b_i, global_mean.
    """
    np.random.seed(seed)
    n_users, n_items = R.shape
    known_mask = ~np.isnan(R)
    known_indices = list(zip(*np.where(known_mask)))

    P = np.random.normal(0, 0.1, (n_users, n_factors))
    Q = np.random.normal(0, 0.1, (n_items, n_factors))
    global_mean = np.nanmean(R)
    b_u = np.zeros(n_users)
    b_i = np.zeros(n_items)

    for epoch in range(n_epochs):
        np.random.shuffle(known_indices)
        for u, i in known_indices:
            pred = global_mean + b_u[u] + b_i[i] + P[u] @ Q[i]
            err = R[u, i] - pred

            b_u[u] += lr * (err - reg * b_u[u])
            b_i[i] += lr * (err - reg * b_i[i])
            p_u_old = P[u].copy()
            P[u] += lr * (err * Q[i] - reg * P[u])
            Q[i] += lr * (err * p_u_old - reg * Q[i])

    return P, Q, b_u, b_i, global_mean

### 5.7.2 Использование готовых библиотек — `surprise` (explicit) и `implicit` (implicit ALS/BPR)

In [ ]:
# --- SVD/Funk SVD через surprise ---
from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings_df[['user_id', 'item_id', 'rating']], reader)

algo = SVD(n_factors=50, lr_all=0.005, reg_all=0.02, n_epochs=20)
cross_validate(algo, data, measures=['RMSE'], cv=5, verbose=True)

# --- Implicit ALS через implicit ---
import implicit
import scipy.sparse as sp

# confidence-матрица, как в разделе 5.5.4/5.5.5
confidence_matrix = sp.csr_matrix(user_item_confidence)

model = implicit.als.AlternatingLeastSquares(factors=50, regularization=0.1, iterations=20)
model.fit(confidence_matrix)

user_factors = model.user_factors
item_factors = model.item_factors

# --- BPR через implicit ---
bpr_model = implicit.bpr.BayesianPersonalizedRanking(factors=50, regularization=0.01, iterations=100)
bpr_model.fit(confidence_matrix)

### 5.7.3 Полномасштабное сравнение на MovieLens

- Обучить SVD (`surprise`), explicit ALS (ручная реализация или `surprise.SVD` с `biased=False` для чистого MF без bias), implicit ALS и BPR (`implicit`) на MovieLens 1M с temporal split (Модуль 1.7).
- Для explicit-моделей посчитать RMSE (Модуль 8.1). Для implicit-моделей (ALS, BPR) посчитать Precision@10 и NDCG@10 (полная формализация метрик — Модуль 8, но можно использовать функции, написанные в практике Модуля 3-4).
- **Ключевое исследование:** построить график зависимости качества от `n_factors` (например, `[10, 20, 50, 100, 200]`) — при каком значении начинается переобучение (RMSE/качество на train продолжает расти, а на test — падает)? Это прямое практическое исследование bias-variance компромисса, уже теоретически разобранного в разделе 5.4.5.
- Сравнить время обучения explicit ALS (ручная реализация, python-циклы) с временем `implicit.als.AlternatingLeastSquares` (векторизованная, скомпилированная реализация) — ощутимая разница послужит наглядной мотивацией того, зачем в production используются специализированные библиотеки, а не самописный код.

### 5.7.4 Вопросы для самопроверки

1. Почему нельзя просто взять `np.linalg.svd()` напрямую на разреженной user-item матрице с пропусками, заполненными нулями, и почему это технически «сработает» (не вызовет ошибку), но даст неверный результат?
2. В разделе 5.4.5 модель почти идеально запомнила обучающие данные (RMSE->0.028). Если бы вместо `λ=0.02` мы использовали `λ=0`, стало бы переобучение сильнее или слабее, и почему?
3. Объясните своими словами, почему шаг обновления `p_u` в ALS (5.5.1) — это точное аналитическое решение, а аналогичный шаг в Funk SVD (5.4.2) — лишь один маленький шаг в сторону оптимума. Как это связано с разницей между «замкнутой формой» (closed-form) и «итеративной оптимизацией» (gradient descent), с которыми вы уже сталкивались при сравнении, например, обычной линейной регрессии (`np.linalg.lstsq`) и её обучения через SGD?
4. В разделе 5.5.4 объясняется вычислительный трюк `Qᵀ·C_u·Q = QᵀQ + Qᵀ(C_u-I)Q`. Почему именно `(C_u - I)`, а не просто `C_u`, оказывается разреженной (в смысле «ненулевой только для реально просмотренных товаров») матрицей?
5. Чем принципиально отличается то, что оптимизирует BPR (5.6.2), от того, что оптимизирует implicit ALS (5.5.4), если оба метода предназначены для одного и того же типа данных (implicit feedback)?

## Глоссарий модуля 5

| Термин | Короткое определение |
|:---|:---|
| Латентный фактор (Latent Factor) | Ненаблюдаемое скрытое измерение, обучаемое автоматически из данных |
| SVD | Точное матричное разложение `R = UΣVᵀ`; требует полностью заполненной матрицы |
| Truncated SVD | Усечение до `k` крупнейших сингулярных чисел — наилучшее приближение ранга `k` |
| Funk SVD | Обучение латентных факторов градиентным спуском только по известным ячейкам |
| Bias-термы (`b_u`, `b_i`) | Систематические смещения пользователя/товара, вынесенные отдельно от латентных векторов |
| ALS | Поочерёдное точное решение задачи наименьших квадратов для `P`, затем для `Q` |
| Confidence (`c_ui`) | Мера доверия к implicit-сигналу, а не сам сигнал предпочтения |
| BPR | Pairwise-метод, напрямую оптимизирующий относительный порядок предпочтений |
| Pointwise / Pairwise | Оптимизация отдельных значений vs оптимизация относительного порядка пар |

**Связь со следующим модулем:** Модуль 6 временно возвращается к более простому, не требующему обучения на взаимодействиях подходу — контентной фильтрации. Это осознанный шаг назад по сложности математики, но не по важности: контентная фильтрация решает именно то, с чем не справляется матричная факторизация — рекомендации для абсолютно нового товара, у которого пока нет вообще ни одного взаимодействия (холодный старт товара, Модуль 1.5.1), поскольку без единого известного `r_ui` обучить `q_i` попросту не из чего.